# 🏢 CORP-ENV · NVIDIA Nemotron-3-Nano-30B-A3B (NVFP4) — SFT + RLVR Training

**End-to-end reproducible notebook** for training the Nemotron-3-Nano-30B model on CORP-ENV using Supervised Fine-Tuning (SFT) followed by Rejection-Sampling RL on Verifiable Rewards (RLVR).

This notebook uses the **general** training scripts (`train_sft_genral.py` / `train_rlvr_genral.py`) which download training data from Hugging Face datasets rather than local files.

| Component | Detail |
|---|---|
| **Base model** | `unsloth/NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4` |
| **SFT script** | `training/train_sft_genral.py` |
| **RLVR script** | `training/train_rlvr_genral.py` |
| **Dataset source** | `Navigam/corp-env-data` (HF Datasets) |
| **Tasks** | E1 Launch Readiness, M1 Budget Reallocation, H1 Acquisition Defence |
| **Runtime** | Lightning AI H100 / A100 (30B model requires ≥40GB VRAM) |

---

## 1️⃣ Setup & Installation

In [ ]:
import os

# ===== Configuration =====
REPO_URL = "https://huggingface.co/spaces/Navigam1108/corp-env"  # Change to your repo
BASE_MODEL = "unsloth/NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4"
HF_ORG_OR_USER = "Navigam1108"  # Your HF username/org
DATASET_REPO = "Navigam/corp-env-data"  # HF dataset repo
DATASET_FILE = "e1_m1_h1_examples.jsonl"

# SFT hyperparameters
SFT_MAX_STEPS = 30        # Quick judge smoke; set -1 for full-epoch training
SFT_EPOCHS = 2.0
SFT_LR = 2e-4
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 8
SFT_MAX_SEQ_LEN = 4096

# RLVR hyperparameters
RLVR_ROUNDS = 3
RLVR_MAX_PROMPTS = 64
RLVR_N_SAMPLES = 8
RLVR_TEMPERATURE = 0.7

# Eval
EVAL_EPISODES = 3
EVAL_MAX_STEPS = 30

# Output paths
SFT_OUTPUT = "outputs/sft_nemotron_nano_30b"
RLVR_OUTPUT = "outputs/rlvr_nemotron_nano_30b"

In [ ]:
# Clone and install
!git clone {REPO_URL} corp_gym 2>/dev/null || echo 'Repo already cloned'
%cd corp_gym
!pip install -U pip
!pip install -e ".[training,plots]"

## 2️⃣ Hugging Face Login

Required for downloading the dataset from `Navigam/corp-env-data` and pushing adapters.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3️⃣ Environment Validation

In [ ]:
!python -m pytest tests/ -v --tb=short 2>/dev/null || echo 'Tests skipped'
!openenv validate

## 4️⃣ Baseline Evaluation

Run scripted (weak) and oracle policies for comparison baselines.

In [ ]:
!python eval.py --policy scripted_weak --label baseline --episodes {EVAL_EPISODES}
!python eval.py --policy oracle --label oracle --episodes {EVAL_EPISODES}

## 5️⃣ SFT Training (Unsloth + TRL)

Fine-tune Nemotron-3-Nano-30B with LoRA using trajectories downloaded from `Navigam/corp-env-data`.

- Uses `training/train_sft_genral.py` — the **general** HF-dataset-compatible SFT script
- Downloads JSONL from HF dataset repo automatically
- LoRA `r=16`, targets attention + MLP projections  
- 4-bit quantization via Unsloth

In [ ]:
!python training/train_sft_genral.py \
  --model {BASE_MODEL} \
  --dataset-repo {DATASET_REPO} \
  --dataset-file {DATASET_FILE} \
  --output {SFT_OUTPUT} \
  --max-steps {SFT_MAX_STEPS} \
  --epochs {SFT_EPOCHS} \
  --lr {SFT_LR} \
  --batch-size {SFT_BATCH_SIZE} \
  --grad-accum {SFT_GRAD_ACCUM} \
  --max-seq-length {SFT_MAX_SEQ_LEN} \
  --hf-user {HF_ORG_OR_USER}

## 6️⃣ Evaluate SFT Adapter

In [ ]:
!python eval.py \
  --policy hf \
  --label sft-nemotron-30b \
  --model {BASE_MODEL} \
  --adapter {SFT_OUTPUT} \
  --episodes {EVAL_EPISODES} \
  --max-steps {EVAL_MAX_STEPS}

## 7️⃣ RLVR Training (Rejection-Sampling FT)

- Uses `training/train_rlvr_genral.py` — the **general** HF-dataset-compatible RLVR script
- Downloads verified trajectory data from HF dataset repo
- Samples completions, scores with `CorpEnvironment` verifier
- Keeps best completions and SFTs on winners

In [ ]:
!python training/train_rlvr_genral.py \
  --model {BASE_MODEL} \
  --adapter {SFT_OUTPUT} \
  --dataset-repo {DATASET_REPO} \
  --examples-files data/processed/e1_m1_clean.jsonl,data/processed/h1_seed_clean.jsonl \
  --output {RLVR_OUTPUT} \
  --rounds {RLVR_ROUNDS} \
  --n-samples {RLVR_N_SAMPLES} \
  --temperature {RLVR_TEMPERATURE} \
  --max-prompts {RLVR_MAX_PROMPTS} \
  --strict-json \
  --use-stub-workers \
  --disable-llm-judge \
  --stats-file results/runs/rlvr_nemotron_30b_stats.jsonl \
  --hf-user {HF_ORG_OR_USER}

## 8️⃣ Evaluate RLVR Adapter

In [ ]:
!python eval.py \
  --policy hf \
  --label rlvr-nemotron-30b \
  --model {BASE_MODEL} \
  --adapter {RLVR_OUTPUT} \
  --episodes {EVAL_EPISODES} \
  --max-steps {EVAL_MAX_STEPS}

## 📊 Generate Comparison Plots

In [ ]:
!python plot_results.py \
  --inputs results/runs \
  --output-dir results/model_compare_nemotron_30b

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path

plot_dir = Path("results/model_compare_nemotron_30b")

for png in sorted(plot_dir.glob("*.png")):
    display(Markdown(f"### {png.stem.replace('_', ' ').title()}"))
    display(Image(filename=str(png), width=800))

# Show summary table
summary_md = plot_dir / "comparison_summary.md"
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

## 📋 Results Summary

Results for `unsloth/NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4` on CORP-ENV:

| Stage | E1 Terminal Reward | M1 Terminal Reward | H1 Terminal Reward | M1 Success |
|-------|-------------------|-------------------|-------------------|------------|
| Baseline (weak) | 0.590 | 0.198 | 0.257 | 0% |
| SFT | *pending* | *pending* | *pending* | *pending* |
| RLVR | *pending* | *pending* | *pending* | *pending* |

> **Note**: Results will be updated after training runs complete. The Nemotron-3-Nano-30B model with NVFP4 quantization provides an interesting test of whether a larger but more aggressively quantized model can outperform the smaller Qwen 2.5-7B on long-horizon planning tasks.